# Dataset et DataLoader : donner les exemples par petits lots

Retrouver un couple mesures/cible, puis comprendre comment il rejoint un lot.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#dataset-et-batches).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


Notre première boucle avait les trois extraits sous la main. Avec une grande sonothèque, nous voudrons retrouver un exemple, puis n’en présenter que quelques-uns au modèle. Commençons par organiser les mêmes trois lignes : rien ne change encore dans leurs mesures ou leurs cibles.

Un Dataset répond à deux questions : combien d’exemples existe-t-il, et que contient celui d’un indice donné ? La classe Extraits conserve X et cibles. __len__ compte les extraits, pas les secondes ; __getitem__ renvoie ensemble une ligne et sa bonne annotation.

Lire dataset[1] rend donc les mesures [0,20 ; 4,5] et la cible 1 de la percussion B. Pour faire un premier lot à la main, prenons A et B : chacun fournit un vecteur de forme (2,).

stack ajoute l’axe qui compte ces deux extraits : le résultat est (2,2). Nous n’avons pas collé les mesures de A derrière celles de B dans un seul vecteur. Le DataLoader effectue cet assemblage pour les exemples de même forme.

batch_size=2 demande au plus deux exemples par lot. shuffle=False laisse A puis B pour vérifier l’ordre ; avec True, les couples seraient mélangés sans séparer leurs cibles. iter ouvre un parcours et next en prend le prochain lot. num_workers=0 garde ici la lecture dans le processus courant.

| Lot | Mesures : forme | Cibles |
| --- | --- | --- |
| Premier | (2,2) | [0,1] |
| Dernier | (1,2) | [0] |

Le dernier lot garde un axe des extraits, même lorsqu’il n’en contient qu’un.

Le dernier lot n’est pas complété artificiellement : drop_last=False, la valeur par défaut, le conserve. Une boucle for entrees, annotations in loader les parcourrait successivement, comme dans un entraînement par minibatches.

Pour réunir ces deux lots, cat allonge l’axe existant : 2+1 lignes, soit (3,2). Les autres tailles doivent correspondre. stack créerait un axe supplémentaire et exigerait des tailles identiques ; les deux opérations répondent à des besoins différents.

## Démonstration 1 · Un Dataset, un empilement et deux lots

Retrouver les objets et exécuter les fragments du chapitre dans leur contexte complet.

### Préparer les bibliothèques

Les trois lignes et leurs annotations restent alignées. La classe définit leur accès.

In [3]:
import torch
from torch.utils.data import Dataset, DataLoader


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [4]:
X = torch.tensor([[.1, 1.5], [.2, 4.5], [.15, 2.5]])
cibles = torch.tensor([0, 1, 0], dtype=torch.long)



In [5]:
observer('X', X)
observer('cibles', cibles)

X forme (3, 2) type torch.float32 requires_grad False
  valeurs : [[0.10000000149011612, 1.5], [0.20000000298023224, 4.5], [0.15000000596046448, 2.5]]
cibles forme (3,) type torch.int64 requires_grad False
  valeurs : [0, 1, 0]


### Définir la règle avant de l’utiliser

Cette cellule définit un objet ou une fonction. Son corps ne sera utilisé que lors d’un appel. Nous gardons la définition entière pour ne pas casser la syntaxe.

In [6]:
class Extraits(Dataset):
    def __init__(self, mesures, annotations):
        self.mesures, self.annotations = mesures, annotations
    def __len__(self):
        return len(self.annotations)
    def __getitem__(self, indice):
        return self.mesures[indice], self.annotations[indice]



### Calculer l’état suivant

Deux vecteurs deviennent deux lignes grâce au nouvel axe de stack.

In [7]:
dataset = Extraits(X, cibles)
mesures_a, cible_a = dataset[0]
mesures_b, cible_b = dataset[1]
lot_manuel = torch.stack([mesures_a, mesures_b], dim=0)


In [8]:
observer('mesures_a', mesures_a)
observer('cible_a', cible_a)
observer('mesures_b', mesures_b)
observer('cible_b', cible_b)
observer('lot_manuel', lot_manuel)

mesures_a forme (2,) type torch.float32 requires_grad False
  valeurs : [0.10000000149011612, 1.5]
cible_a forme () type torch.int64 requires_grad False
  valeurs : 0
mesures_b forme (2,) type torch.float32 requires_grad False
  valeurs : [0.20000000298023224, 4.5]
cible_b forme () type torch.int64 requires_grad False
  valeurs : 1
lot_manuel forme (2, 2) type torch.float32 requires_grad False
  valeurs : [[0.10000000149011612, 1.5], [0.20000000298023224, 4.5]]


### Calculer l’état suivant

Le chargeur retourne les entrées et leurs cibles comme une paire.

In [9]:
loader = DataLoader(dataset, batch_size=2, shuffle=False, num_workers=0)
parcours = iter(loader)
entrees, annotations = next(parcours)


In [10]:
observer('parcours', parcours)
observer('entrees', entrees)
observer('annotations', annotations)

entrees forme (2, 2) type torch.float32 requires_grad False
  valeurs : [[0.10000000149011612, 1.5], [0.20000000298023224, 4.5]]
annotations forme (2,) type torch.int64 requires_grad False
  valeurs : [0, 1]


### Calculer l’état suivant

Un seul exemple reste. cat allonge son axe des extraits pour retrouver le tableau initial.

In [11]:
dernieres_entrees, dernieres_cibles = next(parcours)
reuni = torch.cat([entrees, dernieres_entrees], dim=0)


In [12]:
observer('dernieres_entrees', dernieres_entrees)
observer('dernieres_cibles', dernieres_cibles)
observer('reuni', reuni)

dernieres_entrees forme (1, 2) type torch.float32 requires_grad False
  valeurs : [[0.15000000596046448, 2.5]]
dernieres_cibles forme (1,) type torch.int64 requires_grad False
  valeurs : [0]
reuni forme (3, 2) type torch.float32 requires_grad False
  valeurs : [[0.10000000149011612, 1.5], [0.20000000298023224, 4.5], [0.15000000596046448, 2.5]]


### Lire les sorties

Les sorties vérifient les deux lots et la reconstruction de X.

In [13]:
print("lot manuel :", lot_manuel)
print("premier lot :", tuple(entrees.shape), annotations.tolist())
print("dernier lot :", tuple(dernieres_entrees.shape), dernieres_cibles.tolist())
print("reconstitue X :", torch.equal(reuni, X))


lot manuel : tensor([[0.1000, 1.5000],
        [0.2000, 4.5000]])
premier lot : (2, 2) [0, 1]
dernier lot : (1, 2) [0]
reconstitue X : True


In [14]:
assert bool(len(dataset) == 3 and tuple(lot_manuel.shape) == (2,2)), 'dataset-et-batches/main: len(dataset) == 3 and tuple(lot_manuel.shape) == (2,2)'
assert bool(torch.equal(lot_manuel, entrees) and annotations.tolist() == [0,1]), 'dataset-et-batches/main: torch.equal(lot_manuel, entrees) and annotations.tolist() == [0,1]'
assert bool(tuple(dernieres_entrees.shape) == (1,2) and dernieres_cibles.tolist() == [0]), 'dataset-et-batches/main: tuple(dernieres_entrees.shape) == (1,2) and dernieres_cibles.tolist() == [0]'
assert bool(torch.equal(reuni, X) and cibles.tolist() == [0,1,0]), 'dataset-et-batches/main: torch.equal(reuni, X) and cibles.tolist() == [0,1,0]'
print("Contrôles numériques de la démonstration réussis.")

Contrôles numériques de la démonstration réussis.


## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Ce jeu contient trois paires. Combien d’exemples contient-il ?

In [15]:
exo_x = torch.tensor([[.1,1.5],[.2,4.5],[.15,2.5]])
exo_y = torch.tensor([0,1,0])
exo_dataset = torch.utils.data.TensorDataset(exo_x,exo_y)
exo_nombre = ...  # Remplacer les points par votre réponse.
if exo_nombre is Ellipsis or exo_nombre is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_nombre == len(exo_dataset)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

Chaque paire mesures/cible compte pour un exemple. La largeur des mesures ne compte pas les exemples.

</details>

### Exercice 2

Avec batch_size=2 et drop_last=False, quelle forme aura le dernier lot de mesures ?

In [16]:
exo_x = torch.zeros(3,2)
exo_y = torch.tensor([0,1,0])
exo_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(exo_x,exo_y),batch_size=2,shuffle=False)
exo_forme = ...  # Remplacer les points par votre réponse.
if exo_forme is Ellipsis or exo_forme is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert tuple(exo_forme) == tuple(list(exo_loader)[-1][0].shape)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

Après le premier lot de deux exemples, il reste un exemple avec ses deux caractéristiques.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_x = torch.tensor([[.1,1.5],[.2,4.5],[.15,2.5]])
exo_y = torch.tensor([0,1,0])
exo_dataset = torch.utils.data.TensorDataset(exo_x,exo_y)
exo_nombre = 3
if exo_nombre is Ellipsis or exo_nombre is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_nombre == len(exo_dataset)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_x = torch.zeros(3,2)
exo_y = torch.tensor([0,1,0])
exo_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(exo_x,exo_y),batch_size=2,shuffle=False)
exo_forme = (1,2)
if exo_forme is Ellipsis or exo_forme is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert tuple(exo_forme) == tuple(list(exo_loader)[-1][0].shape)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [19]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()